# Step 1: LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Step 2: Operating Model

1. Run the Drive mount code cell. This is a Colab platform exception, not experiment logic.
2. Run the GCS auth code cell so the runner can stage the fixed pilot subset from public WOMD GCS.
3. Bootstrap the repository and bind artifacts to Drive-backed storage.
4. Stage the fixed 10-shard pilot from standard WOMD v1.1.0 validation tf_example shards into the Drive-bound `assets/raw_womd` cache.
5. Preprocess that staged pilot subset once, then archive it once.
6. In future Colab sessions, restore the pilot archive before running pretrained model evaluations.
7. Apply causal-semantic / interactive filtering after rollout using metadata joins, not by changing the TFRecord source split.


## Step 3: Mount Google Drive

Mount Drive once per Colab runtime so checkpoints, preprocessed artifacts, run bundles, and debug outputs can bind to the persistent project folder.


In [1]:
from pathlib import Path

DRIVE_MOUNTPOINT = "/content/drive"
drive_root = Path(DRIVE_MOUNTPOINT) / "MyDrive"

if drive_root.is_dir():
    print(f"Drive already mounted at {drive_root}")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNTPOINT)
    print(f"Mounted Drive at {drive_root}")


Mounted at /content/drive
Mounted Drive at /content/drive/MyDrive


## Step 4: Authenticate GCS Access

Authenticate the Colab runtime for direct `gs://waymo_open_dataset_motion_v_1_1_0` reads. Keep this enabled unless you intentionally switch to the Drive-local raw WOMD cache.


In [2]:
from __future__ import annotations

import json
import subprocess

USE_GCS_DIRECT = True

if USE_GCS_DIRECT:
    from google.colab import auth
    auth.authenticate_user()
    checks = {}
    commands = {
        "active_account": ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=json"],
        "adc_token": ["gcloud", "auth", "application-default", "print-access-token"],
    }
    for name, command in commands.items():
        proc = subprocess.run(command, text=True, capture_output=True, check=False)
        payload = {"command": command, "returncode": proc.returncode}
        if name == "adc_token":
            payload["token_ready"] = proc.returncode == 0 and bool(proc.stdout.strip())
            payload["stderr"] = proc.stderr.strip()
        else:
            payload["stdout"] = proc.stdout.strip()
            payload["stderr"] = proc.stderr.strip()
        checks[name] = payload
    print(json.dumps({"use_gcs_direct": True, "checks": checks}, indent=2))
else:
    print(json.dumps({
        "use_gcs_direct": False,
        "message": "Skipping GCS auth because this session will use the Drive-local assets/raw_womd cache.",
    }, indent=2))


{
  "use_gcs_direct": true,
  "checks": {
    "active_account": {
      "command": [
        "gcloud",
        "auth",
        "list",
        "--filter=status:ACTIVE",
        "--format=json"
      ],
      "returncode": 0,
      "stdout": "[\n  {\n    \"account\": \"morang.achyut@gmail.com\",\n    \"status\": \"ACTIVE\"\n  }\n]",
      "stderr": ""
    },
    "adc_token": {
      "command": [
        "gcloud",
        "auth",
        "application-default",
        "print-access-token"
      ],
      "returncode": 0,
      "token_ready": true,
      "stderr": ""
    }
  }
}


## Step 5: Bootstrap Repository And Bind Artifacts

Clone or fast-forward `main`, validate the Drive mount, and bind the repository artifact paths to persistent Drive-backed storage.


In [3]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


[colab-bootstrap] $ git clone --branch main https://github.com/achyutmorang/latentdriver-waymax-experiments.git /content/latentdriver-waymax-experiments
{
  "drive_binding": {
    "checkpoints": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/checkpoints",
    "debug_runs": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/debug_runs",
    "drive_root": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments",
    "preprocessed": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/preprocessed",
    "raw_womd": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/raw_womd",
    "results": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/results/runs",
    "smoke": "/content/drive/MyDrive/waymax_research/latentdriver_waymax_experiments/assets/smoke"
  },
  "drive_mount": {
    "mode": "already_mounted",
    "mounted": true,
    "mountpoint": "/

Cloning into '/content/latentdriver-waymax-experiments'...


## Step 6: List Available Runner Profiles

Print the supported `colab_canary.py` profiles. Use this when you need to confirm the exact profile names before launching a run.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


## Step 7: Confirm Git Revision

Fast-forward the local Colab checkout and print the active commit. This is a lightweight sanity check before expensive runs.


In [4]:
%%bash
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main
git rev-parse --short HEAD


Already up to date.
b0aa793


From https://github.com/achyutmorang/latentdriver-waymax-experiments
 * branch            main       -> FETCH_HEAD


## Step 8: Stage Fixed 10-Shard Pilot Dataset

Materialize the fixed dense 10-shard pilot subset into `artifacts/assets/raw_womd/validation_pilot/...`. The source is standard WOMD v1.1.0 `tf_example/validation`, because this repo's vendored Waymax parser expects the standard `roadgraph_samples/*` tf.Example schema. Interaction-focused analysis is applied after rollout through WOMD-Reasoning / CausalAgents metadata, not by using the incompatible `scenario/validation_interactive` split.


In [5]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

# This profile stages from the configured standard WOMD v1.1.0 validation tf_example source,
# overwrites stale pilot shards, and checks for the expected roadgraph_samples/xyz feature.
python3 scripts/colab_canary.py --profile stage-interactive-pilot-shards


Already up to date.

[canary] step 1: stage_interactive_pilot_shards
[canary] $ /usr/bin/python3 scripts/stage_womd_subset_shards.py --source-uri gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord@150 --source-shards 0,15,30,45,60,75,90,105,120,135 --target-uri /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd/validation_pilot/validation_tfexample.tfrecord@10 --force --verify-required-feature roadgraph_samples/xyz
{
  "complete": true,
  "copied": 10,
  "dry_run": false,
  "existing_manifest": null,
  "force": true,
  "manifest_path": "/content/latentdriver-waymax-experiments/artifacts/assets/raw_womd/validation_pilot/validation_tfexample.tfrecord.stage_manifest.json",
  "mappings": [
    {
      "source_index": 0,
      "source_uri": "gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150",
      "target_index": 0,
      "target_uri": "/content/latentdr

From https://github.com/achyutmorang/latentdriver-waymax-experiments
 * branch            main       -> FETCH_HEAD
[stage-womd-subset] staged gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150 -> /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd/validation_pilot/validation_tfexample.tfrecord-00000-of-00010
[stage-womd-subset] staged gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00015-of-00150 -> /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd/validation_pilot/validation_tfexample.tfrecord-00001-of-00010
[stage-womd-subset] staged gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00030-of-00150 -> /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd/validation_pilot/validation_tfexample.tfrecord-00002-of-00010
[stage-womd-subset] staged gs://waymo_open_dataset_

## Step 9: Preprocess The Staged Interactive Pilot Cache

This builds the dedicated pilot preprocess cache under `artifacts/assets/preprocessed/interactive_pilot/...`. Run the status profile before and after the preprocess profile so you can confirm the cache is complete before archiving it.


In [6]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess --auto-install-runtime
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status


Already up to date.
{
  "artifact_status_after_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T192109Z_interactive_pilot_preprocess_status/artifact_status_after.json",
  "artifact_status_before_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T192109Z_interactive_pilot_preprocess_status/artifact_status_before.json",
  "bundle_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T192109Z_interactive_pilot_preprocess_status",
  "debug_aliases": {
    "latest": {
      "alias_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/latest",
      "bundle_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T192109Z_interactive_pilot_preprocess_status",
      "failure_summary_path": null,
      "manifest_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T192109Z_interactive_pilot_preprocess_status/manifest.json",
      "profile": "interactive-pilot-preproc

From https://github.com/achyutmorang/latentdriver-waymax-experiments
 * branch            main       -> FETCH_HEAD
Cloning into '/content/latentdriver-waymax-experiments/external/LatentDriver'...
Note: switching to '721bd6e1f87373457b743d0e0a9606d4d0727b6f'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 721bd6e Update README.md
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7

## Step 10: Create Stable Archive For Future Pilot Evaluations

Once the 10-shard pilot preprocess cache is complete, archive it to Drive-backed storage. This gives you one persistent tar file that future Colab runtimes can restore before evaluating `IDM`, `LatentDriver`, `PlanT`, or your modulated variant on the same fixed pilot subset.


In [8]:
%%bash
cd /content/latentdriver-waymax-experiments
ps -eo pid,etime,cmd | grep -E "preprocess_validation_only|preprocess_data|colab_canary" | grep -v grep || echo "no preprocess process found"


no preprocess process found


In [9]:
%%bash
cd /content/latentdriver-waymax-experiments

MAP=$(find artifacts/assets/preprocessed/interactive_pilot/val_preprocessed_path/map -maxdepth 1 -name "*.npy" 2>/dev/null | wc -l)
ROUTE=$(find artifacts/assets/preprocessed/interactive_pilot/val_preprocessed_path/route -maxdepth 1 -name "*.npy" 2>/dev/null | wc -l)
INT=$(find artifacts/assets/preprocessed/interactive_pilot/val_intention_label -maxdepth 1 -name "*.txt" 2>/dev/null | wc -l)

echo "map=$MAP route=$ROUTE intention=$INT"
test -f artifacts/assets/preprocessed/interactive_pilot/val_preprocessed_path/_SUCCESS && echo "_SUCCESS present" || echo "_SUCCESS missing"
test -f artifacts/assets/preprocessed/interactive_pilot/val_preprocessed_path/preprocess_manifest.json && echo "manifest present" || echo "manifest missing"


map=2899 route=2899 intention=2899
_SUCCESS present
manifest present


In [11]:
%%bash
set -euo pipefail

REPO=/content/latentdriver-waymax-experiments
test -f "$REPO/scripts/colab_canary.py" || {
  echo "repo or scripts/colab_canary.py missing at $REPO"
  echo "Current directory: $(pwd)"
  ls -lah /content
  exit 1
}

cd "$REPO"
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status

{
  "artifact_status_after_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T193448Z_interactive_pilot_preprocess_status/artifact_status_after.json",
  "artifact_status_before_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T193448Z_interactive_pilot_preprocess_status/artifact_status_before.json",
  "bundle_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T193448Z_interactive_pilot_preprocess_status",
  "debug_aliases": {
    "latest": {
      "alias_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/latest",
      "bundle_dir": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T193448Z_interactive_pilot_preprocess_status",
      "failure_summary_path": null,
      "manifest_path": "/content/latentdriver-waymax-experiments/results/debug_runs/20260415T193448Z_interactive_pilot_preprocess_status/manifest.json",
      "profile": "interactive-pilot-preprocess-status",
      "

In [13]:
%%bash
set -euo pipefail

cd /content/latentdriver-waymax-experiments

SRC=artifacts/assets/preprocessed/interactive_pilot
SNAP=/tmp/interactive_pilot_archive_snapshot
TMP=/tmp/interactive_pilot_preprocess_cache.tar
OUT=artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar

rm -rf "$SNAP" "$TMP" artifacts/assets/preprocessed/.interactive_pilot_preprocess_cache.tar.tmp

echo "snapshotting cache to /tmp..."
mkdir -p "$SNAP"
cp -a "$SRC/val_preprocessed_path" "$SRC/val_intention_label" "$SNAP/"

echo "validating snapshot counts..."
MAP=$(find "$SNAP/val_preprocessed_path/map" -maxdepth 1 -name "*.npy" | wc -l)
ROUTE=$(find "$SNAP/val_preprocessed_path/route" -maxdepth 1 -name "*.npy" | wc -l)
INT=$(find "$SNAP/val_intention_label" -maxdepth 1 -name "*.txt" | wc -l)
echo "snapshot map=$MAP route=$ROUTE intention=$INT"

test "$MAP" = "$ROUTE"
test "$MAP" = "$INT"
test -f "$SNAP/val_preprocessed_path/_SUCCESS"
test -f "$SNAP/val_preprocessed_path/preprocess_manifest.json"

echo "creating archive from snapshot..."
tar -C /tmp/interactive_pilot_archive_snapshot \
  -cf "$TMP" \
  val_preprocessed_path \
  val_intention_label

echo "validating archive..."
tar -tf "$TMP" >/dev/null

echo "installing archive..."
mv "$TMP" "$OUT"
ls -lh "$OUT"

echo "checking archive status..."
python3 scripts/preprocess_cache_archive.py status --mode interactive_pilot


snapshotting cache to /tmp...
validating snapshot counts...
snapshot map=2899 route=2899 intention=2899
creating archive from snapshot...
validating archive...
installing archive...
-rw-------+ 1 root root 143M Apr 15 19:40 artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar
checking archive status...
{
  "archive_exists": true,
  "archive_path": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar",
  "archive_size_bytes": 149923840,
  "mode": "interactive_pilot",
  "shard_archive_count": 0,
  "shard_archive_dir": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/interactive_pilot_shard_archives",
  "shard_archive_dir_exists": false,
  "shard_archive_size_bytes": 0,
  "shard_manifest_count": 0,
  "shard_manifest_exists": false,
  "shard_manifest_path": "/content/latentdriver-waymax-experiments/artifacts/assets/preprocessed/interactive_pilot_shard_archives/manifest.json",
  "source_root": "

In [15]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

SRC="artifacts/assets/preprocessed/interactive_pilot"
SNAP="/tmp/interactive_pilot_archive_snapshot_fix"
TMP="/tmp/interactive_pilot_preprocess_cache.tar"
OUT="artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar"

rm -rf "$SNAP" "$TMP"
mkdir -p "$SNAP/interactive_pilot"

cp -a "$SRC/val_preprocessed_path" "$SRC/val_intention_label" "$SNAP/interactive_pilot/"

tar -C "$SNAP" -cf "$TMP" \
  interactive_pilot/val_preprocessed_path \
  interactive_pilot/val_intention_label

mv "$TMP" "$OUT"

echo "archive member layout:"
tar -tf "$OUT" | sed -n '1,20p'

echo
echo "restoring from archive:"
python3 scripts/colab_canary.py --profile restore-interactive-pilot-preprocess-archive

echo
echo "post-restore status:"
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-status


archive member layout:
interactive_pilot/val_preprocessed_path/
interactive_pilot/val_preprocessed_path/preprocess_manifest.json
interactive_pilot/val_preprocessed_path/_SUCCESS
interactive_pilot/val_preprocessed_path/route/
interactive_pilot/val_preprocessed_path/route/536412844.npy
interactive_pilot/val_preprocessed_path/route/1479831763.npy
interactive_pilot/val_preprocessed_path/route/2652836274.npy
interactive_pilot/val_preprocessed_path/route/1402268487.npy
interactive_pilot/val_preprocessed_path/route/1229182461.npy
interactive_pilot/val_preprocessed_path/route/2215556618.npy
interactive_pilot/val_preprocessed_path/route/2020924088.npy
interactive_pilot/val_preprocessed_path/route/2170122321.npy
interactive_pilot/val_preprocessed_path/route/3843547546.npy
interactive_pilot/val_preprocessed_path/route/2243249421.npy
interactive_pilot/val_preprocessed_path/route/2611539564.npy
interactive_pilot/val_preprocessed_path/route/3674106370.npy
interactive_pilot/val_preprocessed_path/rout

## Step 11: Track Interactive Pilot Archive Progress From Colab Terminal

If the archive cell is running, paste this into the Colab terminal to watch the temporary tar file grow, inspect the latest archive log, and confirm when the final archive appears.

```bash
cd /content/latentdriver-waymax-experiments && while true; do clear; date -u; echo; echo "== active archive processes =="; ps -eo pid,etime,cmd | grep -E "interactive_pilot|preprocess_cache_archive|tar -C" | grep -v grep || echo "no archive process found"; echo; echo "== pilot archive files =="; ls -lh artifacts/assets/preprocessed/.interactive_pilot_preprocess_cache.tar.tmp artifacts/assets/preprocessed/interactive_pilot_preprocess_cache.tar 2>/dev/null || echo "archive not created yet"; echo; echo "== latest archive stdout =="; LATEST=$(ls -td results/debug_runs/*create_interactive_pilot_preprocess_archive 2>/dev/null | head -n 1); echo "${LATEST:-no archive debug run yet}"; [ -n "${LATEST:-}" ] && tail -n 80 "$LATEST/steps/01_create_interactive_pilot_preprocess_archive.stdout.log" 2>/dev/null; sleep 60; done
```


## Step 12: Restore The Pilot Archive In Future Colab Sessions

In later Colab runs, restore the archived interactive pilot cache before launching pretrained model evaluations. This should be the first cache step in any fresh runtime after Drive mount, GCS auth, and repository bootstrap.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status
python3 scripts/colab_canary.py --profile restore-interactive-pilot-preprocess-archive
python3 scripts/colab_canary.py --profile interactive-pilot-preprocess-archive-status


## Step 13: Next: Run The 10-Shard Paper-Style Baseline Pair

Run the paired pilot evaluation on the fixed 10-shard subset after restoring the archive. Bootstrap the upstream LatentDriver repo first, then start with the reactive baseline and finish with the non-reactive baseline on the same subset. This is the paper-style baseline pair for the repo's validation_pilot milestone.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
git pull --ff-only origin main

python3 scripts/bootstrap_upstream.py

# Next milestone: paper-style baseline pair on the fixed 10-shard pilot subset for LatentDriver T2-J3.
# Reactive (IDM) baseline first, then non-reactive (expert) baseline on the same staged pilot shards.
# If Colab disconnects, rerun this same cell; the restored archive and staged pilot shards remain reusable.
python3 scripts/check_eval_inputs.py \
  --model latentdriver_t2_j3 \
  --tier interactive_pilot_reactive

python3 scripts/run_waymax_eval.py \
  --model latentdriver_t2_j3 \
  --tier interactive_pilot_reactive \
  --seed 0 \
  --vis false \
  --resumable

python3 scripts/run_waymax_eval.py \
  --model latentdriver_t2_j3 \
  --tier interactive_pilot_non_reactive \
  --seed 0 \
  --vis false \
  --resumable


## Step 14: Inspect Latest Debug Bundle

Use this after a failed canary run. It prints the latest run manifest and latest failure summary without requiring manual traceback hunting.


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Use this after any failed canary run.
python3 - <<'PY'
import json
from pathlib import Path

latest = Path("results/debug_runs/latest/manifest.json")
latest_failure = Path("results/debug_runs/latest_failure/failure_summary.json")
for path in (latest, latest_failure):
    print(f"\n== {path} ==")
    if path.exists():
        print(json.dumps(json.loads(path.read_text()), indent=2)[:12000])
    else:
        print("missing")
PY
